# 00 · Calidad de datos del scraper (EDA de control)

Antes de entrenar el modelo A (`01_price_prediction.ipynb`) conviene saber si los datos scrapeados sirven para eso.
Este notebook corre los mismos seis chequeos que la página **Calidad de datos** de la interfaz
(<http://localhost:3210/calidad>); el código está en `ml/calidad.py` y usa las reglas de limpieza de `ml/precio.py`,
así que audita exactamente los datos con los que se entrena. Documentación: `docs/calidad_datos.md`.

| # | Chequeo | Pregunta |
|---|---|---|
| 0 | Controles básicos | ¿Cada aviso se guardó una vez, completo, con formato y rango esperables y coherente con su URL? |
| 1 | Embudo de limpieza | ¿Qué descarta la limpieza y lo hace parejo entre barrios? |
| 2 | Duplicados y fuga | ¿Cuántas veces aparece la misma unidad y cuánto infla eso las métricas? |
| 3 | Faltantes | ¿Faltan datos al azar o según el precio (MNAR)? |
| 4 | Representatividad | ¿Hay avisos suficientes por barrio para estimar su precio típico con confianza? |
| 5 | Robustez del scraper | ¿El parser extrajo lo mismo durante todo el scrape? ¿Cuánto se cubrió? |

Cada chequeo devuelve un **estado** (OK / Atención / Crítico: el problema en los datos), **hallazgos**, una nota de
**qué implica para el modelo** (y si el pipeline ya lo mitiga), tablas y gráficos.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'ml'))
import pandas as pd
from IPython.display import display, Markdown
import calidad, zpml
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
pd.set_option('display.max_colwidth', 80)

P = calidad.params(dict(operacion='venta', tipo='departamento', duplicados='v2_direccion_texto', excluir_pozo=True,
                        min_barrio=30, umbral_deriva_pp=15, ventana='auto', seed=42))

def mostrar(s):
    """Muestra una sección: estado, KPIs, hallazgos, implicancia, tablas y gráficos."""
    estado = {'ok': '✅ OK', 'atencion': '⚠️ Atención', 'critico': '❌ Crítico', 'sin_datos': '— Sin datos'}[s['estado']]
    txt = f"### {s['titulo']} — {estado}\n**{s['pregunta']}**\n\n"
    txt += ' · '.join(f"{k['label']}: **{k['valor']}**" + (f" ({k['nota']})" if k['nota'] else '') for k in s['kpis']) + '\n\n'
    txt += '\n'.join(f'- {h}' for h in s['hallazgos'])
    if s['mitigacion']:
        txt += f"\n\n> **Qué implica para el modelo:** {s['mitigacion']}"
    display(Markdown(txt))
    for fig in s['figuras'].values():
        display(fig)
    for nombre, t in s['tablas'].items():
        display(Markdown(f'**{nombre}**'))
        display(t.head(25))

## Datos

Avisos de CABA como los ve el modelo (`zpml.load()`), todas las líneas de `listings.jsonl` (con historial), `frontier.json`, y `failures.jsonl`.

In [ ]:
X = calidad.cargar()
df, raw = X['df'], X['raw']
print(f"{len(raw):,} líneas en listings.jsonl · {raw['id'].nunique():,} ids · {len(df):,} avisos de CABA")
print('frontier.json:', 'sí' if X['frontier'] else 'no', '· failures.jsonl:', 0 if X['failures'] is None else len(X['failures']))

## 0. Controles básicos del scraper

Sobre **todos** los avisos guardados (venta, alquiler y temporario; última versión de cada id), sin ninguna regla del
modelo: completitud por campo y operación, formato (id, URL, fechas, moneda, enteros), unicidad (líneas y URLs
repetidas), rangos de plausibilidad, consistencia entre campos (dormitorios ≤ ambientes, operación de la búsqueda) y
concordancia con la URL del aviso, que es una fuente independiente del texto de la tarjeta.

In [ ]:
s0 = calidad.basicos(raw, P)
mostrar(s0)

## 1. Embudo de limpieza

Las reglas de `precio.limpiar()` (moneda, superficie, ambientes, rangos de precio y USD/m², pozo, duplicados) más el
recorte de USD/m² en train. Se registra qué paso descartó cada aviso para ver si la limpieza **cambia el peso de algún
barrio**: si lo hace, el modelo aprende sobre una muestra distinta de la publicada.

In [ ]:
s1, d = calidad.embudo(df, P)
mostrar(s1)

## 2. Duplicados y fuga entre train y test

La misma unidad aparece publicada por varias inmobiliarias o republicada con otro id. Si una copia queda en train y otra
en test, el modelo «recuerda» el precio y la métrica sale optimista. Se compara el porcentaje de test con un gemelo en
train según el protocolo de separación.

In [ ]:
s2 = calidad.duplicados(df, raw, P)
mostrar(s2)

## 3. Faltantes: ¿al azar o según el precio?

Para cada campo se compara el USD/m² **relativo a la mediana del barrio** (así no se confunde con el efecto barrio) entre
avisos con y sin el dato (Mann-Whitney). Si falta más en avisos baratos, el faltante es informativo (MNAR) y no conviene
imputarlo con un valor fijo sin indicador.

In [ ]:
s3 = calidad.faltantes(df, d, P)
mostrar(s3)

## 4. Representatividad por barrio

Avisos por barrio después de la limpieza e incertidumbre de la mediana de USD/m² (IC95 por bootstrap, crece como 1/√n):
con pocos avisos, la referencia de precio del barrio que usa el modelo es inestable.

In [ ]:
s4 = calidad.representatividad(d, P)
mostrar(s4)

## 5. Robustez del scraper: deriva del parser y cobertura

Específico de web mining: un cambio de HTML o una página degradada no da error, deja campos vacíos. Se mide el % de
campos vacíos por franja de tiempo del scraping (`scraped_at`) y se marcan las franjas que se alejan de lo típico; además,
la cobertura de cada búsqueda (guardados / anunciados por ZonaProp, de `frontier.json`), las cargas fallidas y la validez
de formatos (moneda, ubicación, barrio, tipo).

In [ ]:
s5 = calidad.scraper(raw, X['frontier'], X['failures'], X['settings'], P)
mostrar(s5)

## Resumen

In [ ]:
calidad.resumen([s0, s1, s2, s3, s4, s5])